# Gimlet Heterogeneous Precision/Width — Kaggle Proof Runbook

This notebook mirrors the working Colab proof flow for Kaggle.

Recommended order:
1. Setup
2. Minimal patch
3. 300-step proof run
4. 600-step run only if 300 succeeds


In [ ]:
# ====== KAGGLE CELL 1: SETUP ======
import os, shutil, subprocess
from pathlib import Path
import torch

WORK = Path('/kaggle/working')
os.chdir(WORK)

for d in ['pg', 'openai-pg']:
    p = WORK / d
    if p.exists():
        shutil.rmtree(p)

print('Cloning repositories...')
subprocess.run([
    'git', 'clone', '-b', 'feat/gimlet-hetero-v1',
    'https://github.com/jmoncayo-pursuit/parameter-golf-gimlet-hetero.git', 'pg'
], check=True)
subprocess.run([
    'git', 'clone', 'https://github.com/openai/parameter-golf.git', 'openai-pg'
], check=True)

print('Installing deps...')
subprocess.run([
    'python3', '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'sentencepiece', 'zstandard', 'huggingface_hub'
], check=True)

print('Downloading tokenizer + 1 train shard...')
os.chdir(WORK / 'openai-pg')
subprocess.run([
    'python3', 'data/cached_challenge_fineweb.py', '--train-shards', '1'
], check=True)

tok = WORK / 'openai-pg' / 'data' / 'tokenizers' / 'fineweb_1024_bpe.model'
print('Tokenizer exists:', tok.exists(), tok)
print('Tokenizer bytes:', tok.stat().st_size if tok.exists() else -1)

print('CUDA Status:')
available = torch.cuda.is_available()
count = torch.cuda.device_count()
print(f'CUDA Available: {available}')
print(f'Device Count: {count}')
if available:
    for i in range(count):
        print(f'Device {i}: {torch.cuda.get_device_name(i)}')

print('Setup complete.')


In [ ]:
# ====== KAGGLE CELL 2: MINIMAL PATCH ======
from pathlib import Path
import subprocess

p = Path('/kaggle/working/pg/train_gpt.py')
text = p.read_text()

# Simplified log0 patch: replace the early call with a print to avoid UnboundLocalError
text = text.replace(
    'log0("Note: Running on CPU (testing only). Training will be extremely slow.")',
    'if rank == 0: print("Note: Running on CPU (testing only). Training will be extremely slow.")'
)

# Add VAL_MAX_TOKENS support
text = text.replace(
    'val_loss_every = int(os.environ.get("VAL_LOSS_EVERY", 1000))',
    'val_loss_every = int(os.environ.get("VAL_LOSS_EVERY", 1000))\n    val_max_tokens = int(os.environ.get("VAL_MAX_TOKENS", 0))'
)

text = text.replace(
    'val_tokens = load_validation_tokens(args.val_files, args.train_seq_len)',
    'val_tokens = load_validation_tokens(args.val_files, args.train_seq_len)\n        if args.val_max_tokens > 0:\n            val_tokens = val_tokens[:min(val_tokens.numel(), args.val_max_tokens + 1)]'
)

# Instrumentation for logging (rely on existing log0 which is defined later in main)
text = text.replace(
    'if master_process:\n        torch.save(base_model.state_dict(), "final_model.pt")',
    'log0(f"phase:training_complete steps:{step}/{args.iterations}")\n    if master_process:\n        log0("phase:serialization_start writing final_model.pt")\n        torch.save(base_model.state_dict(), "final_model.pt")'
)

text = text.replace(
    'quant_obj, quant_stats = quantize_state_dict_int8',
    'log0("phase:quantization_start building int8+zlib artifact")\n    quant_obj, quant_stats = quantize_state_dict_int8'
)

text = text.replace(
    'with open("final_model.int8.ptz", "rb") as f:',
    'log0("phase:roundtrip_eval_start loading quantized artifact and running final validation")\n    with open("final_model.int8.ptz", "rb") as f:'
)

text = text.replace(
    'log0(f"final_int8_zlib_roundtrip_exact',
    'log0("phase:done")\n    log0(f"final_int8_zlib_roundtrip_exact'
)

p.write_text(text)
subprocess.run(['python3', '-m', 'py_compile', str(p)], check=True)
print('Patch applied and syntax OK.')


In [ ]:
# ====== KAGGLE CELL 3: 300-STEP PROOF RUN ======
import os, subprocess, time

os.chdir('/kaggle/working/pg')

env = os.environ.copy()
env.update({
    'NO_COMPILE': '1',
    'ITERATIONS': '300',
    'TRAIN_SEQ_LEN': '128',
    'TRAIN_BATCH_TOKENS': '8192',
    'VAL_LOSS_EVERY': '0',
    'TRAIN_LOG_EVERY': '10',
    'WARMUP_STEPS': '2',
    'MAX_WALLCLOCK_SECONDS': '0',
    'VAL_MAX_TOKENS': '65536',
    'DATA_PATH': '/kaggle/working/openai-pg/data/datasets/fineweb10B_sp1024',
    'TOKENIZER_PATH': '/kaggle/working/openai-pg/data/tokenizers/fineweb_1024_bpe.model',
    'PYTHONUNBUFFERED': '1'
})

print('Starting Kaggle 300-step proof run at', time.strftime('%H:%M:%S'))

proc = subprocess.Popen(
    ['python3', '-u', 'train_gpt.py'],
    cwd='/kaggle/working/pg',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

rc = proc.wait()
print('\nTraining finished with exit code', rc, 'at', time.strftime('%H:%M:%S'))

print('\nArtifact check:')
for name in ['final_model.pt', 'final_model.int8.ptz', 'final_summary.json', 'final_summary.md']:
    path = f'/kaggle/working/pg/{name}'
    if os.path.exists(path):
        print('OK', name, os.path.getsize(path), 'bytes')
    else:
        print('MISSING', name)


In [ ]:
# ====== KAGGLE CELL 4: 600-STEP RUN ======
import os, subprocess, time

os.chdir('/kaggle/working/pg')

env = os.environ.copy()
env.update({
    'NO_COMPILE': '1',
    'ITERATIONS': '600',
    'TRAIN_SEQ_LEN': '128',
    'TRAIN_BATCH_TOKENS': '8192',
    'VAL_LOSS_EVERY': '0',
    'TRAIN_LOG_EVERY': '10',
    'WARMUP_STEPS': '2',
    'MAX_WALLCLOCK_SECONDS': '0',
    'VAL_MAX_TOKENS': '65536',
    'DATA_PATH': '/kaggle/working/openai-pg/data/datasets/fineweb10B_sp1024',
    'TOKENIZER_PATH': '/kaggle/working/openai-pg/data/tokenizers/fineweb_1024_bpe.model',
    'PYTHONUNBUFFERED': '1'
})

print('Starting Kaggle 600-step run at', time.strftime('%H:%M:%S'))

proc = subprocess.Popen(
    ['python3', '-u', 'train_gpt.py'],
    cwd='/kaggle/working/pg',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

rc = proc.wait()
print('\nTraining finished with exit code', rc, 'at', time.strftime('%H:%M:%S'))

print('\nArtifact check:')
for name in ['final_model.pt', 'final_model.int8.ptz', 'final_summary.json', 'final_summary.md']:
    path = f'/kaggle/working/pg/{name}'
    if os.path.exists(path):
        print('OK', name, os.path.getsize(path), 'bytes')
    else:
        print('MISSING', name)
